# 00. Demostración Rápida: El momento "Aha!" de Great Expectations

¡Bienvenidos al taller! Antes de aprender cómo configurar y programar cada detalle paso a paso, vamos a ver **por qué estamos aquí y cuál es el resultado final**.

En esta demo rápida, vamos a tomar un dataset de ventas que es intencionalmente grande y "sucio" (`ventas_sucias.csv` - 1500 filas). Tiene problemas de la vida real:
- Fechas en el futuro
- IDs de clientes nulos
- Precios negativos o en cero
- Errores tipográficos en categorías

Vamos a aplicarle algunas simples reglas de calidad, ¡y ver cómo Great Expectations (GX) genera documentación web interactiva (*Data Docs*) automáticamente que expone todos los problemas visualmente!

In [4]:
import great_expectations as gx
import pandas as pd

### 1. Inicializar un Contexto Efímero y Cargar el Dataset Manchado:
No te preocupes por entender cada línea aún, lo veremos lento en los próximos notebooks.

In [9]:
context = gx.get_context(mode="ephemeral")
df_ventas = pd.read_csv("../data/ventas_sucias.csv")

data_source = context.data_sources.add_pandas(name="ecommerce_data")
data_asset = data_source.add_dataframe_asset(name="ventas_asset")
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch_ventas")

### 2. Escribir Expectativas (Reglas de Calidad)
Aquí declaramos lo que "esperamos" de un dataset sano: precios siempre positivos, la cantidad de items superior a cero, y las categorías no deben tener typos.

In [10]:
# Crear las expectativas
exp_precio = gx.expectations.ExpectColumnValuesToBeBetween(column="price", min_value=0.01)
exp_cantidad = gx.expectations.ExpectColumnValuesToBeBetween(column="quantity", min_value=1, max_value=100)
exp_categoria = gx.expectations.ExpectColumnDistinctValuesToBeInSet(column="product_category", value_set=["Electronics", "Clothing", "Home", "Toys"])
exp_nulos = gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id")

# Guardarlas en una Suite
suite = context.suites.add(gx.ExpectationSuite(name="demostracion_suite_ventas"))
suite.add_expectation(exp_precio)
suite.add_expectation(exp_cantidad)
suite.add_expectation(exp_categoria)
suite.add_expectation(exp_nulos)

ExpectColumnValuesToNotBeNull(id='d51d1b4c-853a-4843-9d9e-b20f84fba5c9', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='customer_id', mostly=1, row_condition=None, condition_parser=None)

### 3. Ejecutar la Validación
Pasamos los 1500 registros por las reglas, sabiendo que fallarán.

In [11]:
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_definition, suite=suite, name="validacion_inicial")
)
validation_result = validation_definition.run(batch_parameters={"dataframe": df_ventas})

print("¿Validación Exitosa?:", validation_result.success)

Calculating Metrics:   0%|          | 0/24 [00:00<?, ?it/s]

¿Validación Exitosa?: False


### 4.Ver los resultados visualmente (Data Docs)
Como ven, la salida fue un amargo `False`. En terminal, ver qué de los 1500 registros falló y por qué sería una pesadilla.
GX soluciona esto generando páginas HTML interactivas donde la Calidad de Datos es transparente tanto para desarrolladores como para negocio.

**Ejecuta esta celda y abrirá una pestaña en tu navegador:**

In [12]:
context.build_data_docs()
context.open_data_docs()

An unexpected Exception occurred during data docs rendering.  Because of this error, certain parts of data docs will not be rendered properly and/or may not appear altogether.  Please use the trace, included in this message, to diagnose and repair the underlying issue.  Detailed information follows:
            KeyError: "'unexpected_percent'".  Traceback: "Traceback (most recent call last):
  File "c:\Users\USER\anaconda3\Lib\site-packages\great_expectations\render\renderer\content_block\validation_results_table_content_block.py", line 141, in row_generator_fn
    unexpected_statement_renderer[1](result=result)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "c:\Users\USER\anaconda3\Lib\site-packages\great_expectations\render\renderer\renderer.py", line 27, in inner_func
    return renderer_fn(*args, **kwargs)
  File "c:\Users\USER\anaconda3\Lib\site-packages\great_expectations\expectations\expectation.py", line 1021, in _diagnostic_unexpected_statement_renderer
    unexpec